# Dataiku 법령 중계 API 확인
첨부 GUIDE_UTF8.txt/DOCX의 순서로 실행하세요. 사내 Dataiku 실제 연결은 실행 후 판정합니다.
외부 공개 API 확인과 사내 중계 확인은 별개입니다. 등록된 OC를 입력창에 넣으세요.
기본 5회 호출, 자동 재시도 0회. 확인 결과는 RAW/MAIN에 자동 반영되지 않습니다.


## 1. 실행 설정

In [ ]:
# [셀 1] 이 설정만 수정합니다. 사진의 주소는 개발계 샘플입니다.
CONFIG = {
    "mode": "relay",  # 사내 Dataiku에서는 relay 유지. 외부 직접 호출로 자동 전환하지 않음
    "relay_origin": "https://45.101.64.86:8443",
    "law_origin": "http://www.law.go.kr",  # 사진의 원본 URL; 중계서버 접속은 HTTPS
    "making_origin": "https://opinion.lawmaking.go.kr",  # 사진1 샘플. 사진2는 www 호스트
    "target_in_header": True,  # 사진2처럼 target_url에 ?target=law 등을 포함
    "ca_bundle": "",  # 사내 CA PEM이 필요하면 Dataiku 실행 서버의 실제 경로 입력
    "law_name": "자본시장과 금융투자업에 관한 법률",
    "law_id": "010513",  # 2026-10-15 외부 API에서 확인한 법령ID. MST와 다름
    "admrul_name": "금융투자업규정",
    "output_root": "law_api_check_results",
    "ip_lookup_url": "https://45.1.81.80:8443/oap/common/query/ip",
}
USE_RELAY_COOKIE = False  # 담당자가 쿠키가 필요하다고 확인한 경우만 True
RUN_IP_LOOKUP = False  # 운영 주소 조회가 필요할 때 별도 실행. 응답의 IP를 자동 선택하지 않음
EXTR_INSN_SECT_CODE = "0"  # 사진의 개발계 예시. 운영 값은 담당자 명세로 확인
RUN_NOTICES = False  # 입법/행정예고 확인까지 필요할 때 True; 별도 OC 입력
MANAGED_FOLDER_ID = ""  # Dataiku에서 만든 law_api_check_results 폴더의 ID 입력


## 2. 공통 함수 — 수정 없이 실행

In [ ]:
# [셀 2] 요청·검증·원문/실패 기록 함수 정의
"""Dataiku 법령 API 수신 진단. 수집/MAIN 등록 도구가 아닌 독립 현장 확인 코드."""
import csv
import gzip
import hashlib
import io
import json
import os
import platform
import re
import sys
import time
import uuid
from pathlib import Path
from urllib.parse import quote, quote_plus, urlsplit
import xml.etree.ElementTree as ET

import requests

VERSION = "dataiku-law-api-20261015-r1"
MAX_BYTES = 12 * 1024 * 1024
MAX_CALLS = 8


def digest(raw):
    return hashlib.sha256(raw).hexdigest()


def json_bytes(obj):
    return (json.dumps(obj, ensure_ascii=False, indent=2) + "\n").encode("utf-8")


def masked(text, secrets=()):
    text = str(text)
    for secret in secrets:
        if secret:
            for value in {secret, quote(secret, safe=""), quote_plus(secret)}:
                text = text.replace(value, "REDACTED")
    text = re.sub(r"(?i)(OC(?:=|%3D))[^&\s<>\"']+", r"\1REDACTED", text)
    return text


def clean_xml(raw):
    if b"<!DOCTYPE" in raw.upper() or b"<!ENTITY" in raw.upper():
        raise ValueError("XML_DTD_OR_ENTITY")
    root = ET.fromstring(raw)
    # Namespace is harmless formatting; original bytes remain separately hashed.
    for elem in root.iter():
        elem.tag = elem.tag.rsplit("}", 1)[-1]
    return root


def first(root, name):
    return (root.findtext(".//" + name) or "").strip()


def normalized(text):
    return re.sub(r"\s+", "", text).replace("·", "ㆍ")


def inspect_xml(raw, kind, expected_name="", expected_id=""):
    root = clean_xml(raw)
    if root.tag.lower() in {"html", "error", "errors"}:
        raise ValueError("HTML_OR_API_ERROR")
    for tag in ("error", "errorCode", "errCode", "오류메시지"):
        if first(root, tag):
            raise ValueError("API_ERROR_XML")
    if kind in ("law_list", "admrul_list"):
        law = kind == "law_list"
        required_root = "LawSearch" if law else "AdmRulSearch"
        if root.tag != required_root:
            raise ValueError("UNEXPECTED_XML_ROOT")
        rows = []
        for el in root.iter("law" if law else "admrul"):
            rows.append({
                "name": first(el, "법령명한글" if law else "행정규칙명"),
                "id": first(el, "법령ID" if law else "행정규칙ID"),
                "serial": first(el, "법령일련번호" if law else "행정규칙일련번호"),
                "current": first(el, "현행연혁코드" if law else "현행연혁구분"),
                "effective_date": first(el, "시행일자"),
            })
        total = first(root, "totalCnt")
        if not total.isdigit() or (int(total) > 0 and not rows):
            raise ValueError("LIST_SCHEMA_MISMATCH")
        return {"status": "LIST_OK" if rows else "EMPTY_LIST", "root": root.tag,
                "total_count": int(total), "rows": rows,
                "note": "목록 수신은 본문 수신/전체 수집 성공과 별개"}
    if kind in ("law_body", "admrul_body"):
        law = kind == "law_body"
        if root.tag != ("법령" if law else "AdmRulService"):
            raise ValueError("UNEXPECTED_XML_ROOT")
        name = first(root, "법령명_한글" if law else "행정규칙명")
        doc_id = first(root, "법령ID" if law else "행정규칙일련번호")
        contents = ["".join(x.itertext()).strip() for x in root.iter("조문내용")]
        if not name or not any(contents) or not doc_id:
            raise ValueError("BODY_MISSING_NAME_ID_OR_ARTICLE")
        if expected_name and normalized(name) != normalized(expected_name):
            raise ValueError("BODY_NAME_MISMATCH")
        if expected_id and doc_id.lstrip("0") != expected_id.lstrip("0"):
            raise ValueError("BODY_ID_MISMATCH")
        return {"status": "BODY_OK", "root": root.tag, "document_name": name, "id": doc_id,
                "effective_date": first(root, "시행일자"),
                "promulgated_date": first(root, "공포일자" if law else "발령일자"),
                "article_blocks": len(contents), "article_characters": sum(map(len, contents)),
                "annex_links": sum(1 for x in root.iter() if x.tag in
                                   {"별표서식파일링크", "첨부파일링크"} and x.text),
                "sample": next(x for x in contents if x)[:240]}
    if kind == "notice":
        # The relay response contract is not supplied. XML alone cannot certify data.
        return {"status": "NOTICE_XML_REVIEW_REQUIRED", "root": root.tag,
                "child_tags": [e.tag for e in list(root)[:15]],
                "note": "예고 API XML 수신만 확인. 목록/0건/오류 판정은 실제 명세와 대조 필요"}
    if kind == "ip":
        raise ValueError("IP_RESPONSE_MUST_BE_JSON")
    raise ValueError("UNKNOWN_RESPONSE_KIND")


class RelayCheck:
    def __init__(self, config, law_oc, making_oc="", cookie="", session=None):
        self.config = dict(config)
        self.oc = law_oc.strip()
        self.making_oc = making_oc.strip()
        self.cookie = cookie.strip()
        self.secrets = (self.oc, self.making_oc, self.cookie)
        if not self.oc or self.oc.lower() == "test":
            raise ValueError("등록된 LAW_OC를 입력하세요. test 예제값은 사용하지 않습니다.")
        if any("\n" in x or "\r" in x for x in self.secrets):
            raise ValueError("인증 입력에 줄바꿈이 있습니다.")
        self.mode = config.get("mode", "relay")
        if self.mode not in ("relay", "direct_public"):
            raise ValueError("mode는 relay 또는 외부 검증용 direct_public만 가능합니다.")
        self.verify = config.get("ca_bundle") or True
        if config.get("ca_bundle") is False or (self.verify is not True and not Path(self.verify).is_file()):
            raise ValueError("CA 파일 경로를 확인하세요. 인증서 검증 해제는 지원하지 않습니다.")
        self.origin = config["relay_origin"].rstrip("/")
        p = urlsplit(self.origin)
        if p.scheme != "https" or not p.hostname or p.username or p.password or p.path or p.query or p.fragment:
            raise ValueError("relay_origin은 https://승인호스트:포트 형식입니다.")
        self.law_origin = config.get("law_origin", "http://www.law.go.kr").rstrip("/")
        if self.law_origin not in ("http://www.law.go.kr", "https://www.law.go.kr"):
            raise ValueError("법제처 원본 호스트 불일치")
        self.making_origin = config.get("making_origin", "https://opinion.lawmaking.go.kr").rstrip("/")
        if self.making_origin not in ("https://opinion.lawmaking.go.kr", "https://www.lawmaking.go.kr"):
            raise ValueError("입법센터 원본 호스트 불일치")
        self.session = session or requests.Session()
        if session is None:
            self.session.mount("https://", requests.adapters.HTTPAdapter(max_retries=0))
            self.session.mount("http://", requests.adapters.HTTPAdapter(max_retries=0))
        self.run_id = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "_" + uuid.uuid4().hex[:8]
        self.out = Path(config.get("output_root", "law_api_check_results")) / self.run_id
        self.out.mkdir(parents=True, exist_ok=False, mode=0o700)
        self.records = {}
        self.started = time.monotonic()
        self.last_call = 0.0
        self.calls = 0
        self.blocked = False
        self._save_summary()

    def _write(self, name, raw):
        path = self.out / name
        with path.open("xb") as handle:
            handle.write(raw)
        path.chmod(0o600)

    def _save_summary(self):
        summary = {"version": VERSION, "run_id": self.run_id, "transport": self.mode,
                   "python": platform.python_version(), "executable": sys.executable,
                   "requests_version": requests.__version__, "physical_calls": self.calls,
                   "max_calls": MAX_CALLS, "automatic_retries": 0,
                   "resume_available": False, "records": list(self.records.values()),
                   "scope": "소량 API 연결 진단. RAW 반입/MAIN/GENOS 배포는 수행하지 않음"}
        tmp = self.out / "summary.tmp"
        tmp.write_bytes(json_bytes(summary))
        tmp.chmod(0o600)
        tmp.replace(self.out / "summary.json")

    def request(self, name, path, target, params, kind, expected_name="", expected_id=""):
        if not re.fullmatch(r"[a-z0-9_]+", name):
            raise ValueError("Invalid diagnostic name")
        if path not in {"lawSearch.do", "lawService.do", "ogLmPp.xml", "ogLmPpMod.xml", "ptcpAdmPp.xml", "ip"}:
            raise ValueError("허용되지 않은 API 경로")
        if path.startswith("law") and target not in {"law", "eflaw", "admrul"}:
            raise ValueError("허용되지 않은 target")
        if any(k.lower() in {"oc", "target", "target_url", "cookie", "authorization"} for k in params):
            raise ValueError("인증과 target은 params에 직접 넣지 않습니다.")
        sig = digest(json_bytes([path, target, params, kind, expected_name, expected_id]))
        if name in self.records:
            if self.records[name]["request_signature"] != sig:
                raise ValueError("같은 이름의 요청 조건이 변경되었습니다. 새 진단을 시작하세요.")
            return self.records[name]  # success AND failure: do not blindly retry a cell
        row = {"name": name, "request_signature": sig, "target": target,
               "params": params, "transport": self.mode, "status": "PENDING",
               "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
        self.records[name] = row
        if self.blocked or self.calls >= MAX_CALLS or time.monotonic() - self.started > 900:
            row["status"] = "BLOCKED_OR_BUDGET_EXHAUSTED"
            self._save_summary()
            return row
        is_making = path.endswith(".xml")
        if is_making and not self.making_oc:
            row["status"] = "MISSING_MAKING_OC"
            self._save_summary()
            return row
        arguments = dict(params)
        method, body = "GET", None
        headers = {"Accept": "application/xml, text/xml, application/json"}
        if path == "ip":
            if self.mode != "relay":
                raise ValueError("IP 조회는 사내 relay 모드에서만 실행")
            url = self.config.get("ip_lookup_url", "https://45.1.81.80:8443/oap/common/query/ip")
            p = urlsplit(url)
            if p.scheme != "https" or not p.hostname or p.username or p.password or p.query or p.fragment or p.path != "/oap/common/query/ip":
                raise ValueError("운영계 IP 조회 API 주소를 확인하세요.")
            method, body = "POST", {"EXTR_INSN_SECT_CODE": str(params.get("EXTR_INSN_SECT_CODE", "0"))}
            arguments = None
            row["original_url"] = url
        else:
            upstream = self.making_origin + "/rest/" + path if is_making else self.law_origin + "/DRF/" + path
            row["original_url"] = upstream + ("?target=" + target if target else "")
            arguments["OC"] = self.making_oc if is_making else self.oc
            if self.mode == "relay":
                url = self.origin + ("/law/making" if is_making else "/law/DRF")
                # target is placed ONCE: in header or query, never both.
                headers["target_url"] = upstream
                if target and self.config.get("target_in_header", True):
                    headers["target_url"] += "?target=" + target
                elif target:
                    arguments["target"] = target
            else:
                url = upstream.replace("http://", "https://", 1)
                if target:
                    arguments["target"] = target
        if self.cookie and self.mode == "relay":
            headers["Cookie"] = self.cookie
        time.sleep(max(0.0, 1.0 - (time.monotonic() - self.last_call)))
        self.last_call = time.monotonic()
        self.calls += 1
        row["physical_call"] = self.calls
        row["status"] = "IN_FLIGHT"
        self._save_summary()  # interrupt leaves an honest IN_FLIGHT record
        start = time.monotonic()
        raw = bytearray()
        try:
            with self.session.request(method, url, params=arguments, json=body, headers=headers,
                                      timeout=(8, 20), verify=self.verify,
                                      allow_redirects=False, stream=True) as response:
                row["http_status"] = response.status_code
                row["content_type"] = masked(response.headers.get("Content-Type", ""), self.secrets)
                for chunk in response.iter_content(65536):
                    if time.monotonic() - start > 45:
                        raise ValueError("RESPONSE_TIME_LIMIT")
                    if len(raw) + len(chunk) > MAX_BYTES:
                        raise ValueError("RESPONSE_SIZE_LIMIT")
                    raw.extend(chunk)
                if 300 <= response.status_code < 400:
                    raise ValueError("REDIRECT_BLOCKED")
                if response.status_code != 200:
                    if response.status_code in (401, 403, 429):
                        self.blocked = True
                    raise ValueError("HTTP_" + str(response.status_code))
                if kind == "ip":
                    parsed = json.loads(bytes(raw))
                    if not isinstance(parsed, (dict, list)):
                        raise ValueError("IP_JSON_SCHEMA_UNKNOWN")
                    row.update(status="IP_RESPONSE_REVIEW_REQUIRED",
                               note="실제 응답은 저장파일로 확인. 운영 주소 선택/성공코드/환경코드는 담당자 명세와 대조")
                else:
                    row.update(inspect_xml(bytes(raw), kind, expected_name, expected_id))
        except requests.exceptions.SSLError:
            self.blocked = True
            row.update(status="TLS_ERROR", reason="인증서/호스트명/CA 확인. 검증을 해제하지 마세요.")
        except requests.exceptions.Timeout:
            self.blocked = True
            row.update(status="TIMEOUT", reason="Dataiku 실행 서버→중계서버 네트워크/업스트림 시간 확인")
        except requests.exceptions.ConnectionError:
            self.blocked = True
            row.update(status="CONNECTION_ERROR", reason="DNS/방화벽/포트/프록시/실행 컨테이너 연결 확인")
        except requests.exceptions.RequestException as exc:
            row.update(status="TRANSPORT_ERROR", error_class=type(exc).__name__)
        except (ET.ParseError, UnicodeError, json.JSONDecodeError) as exc:
            row.update(status="PARSE_ERROR", error_class=type(exc).__name__, reason="XML 대신 HTML/로그인/JSON 포장/잘린 응답인지 확인")
        except ValueError as exc:
            row.update(status="REJECTED", reason=masked(str(exc), self.secrets))
        finally:
            row["elapsed_seconds"] = round(time.monotonic() - start, 3)
            row["received_bytes"] = len(raw)
            row["wire_sha256"] = digest(raw)
            if raw:
                # Search responses may echo OC inside detail URLs: never save that secret.
                clean = bytes(raw)
                for secret in self.secrets:
                    if secret:
                        for value in {secret, quote(secret, safe=""), quote_plus(secret)}:
                            clean = clean.replace(value.encode(), b"REDACTED")
                row["credential_redacted"] = clean != raw
                row["saved_sha256"] = digest(clean)
                row["response_file"] = name + ".response.gz"
                self._write(row["response_file"], gzip.compress(clean, mtime=0))
            safe_row = json.loads(masked(json.dumps(row, ensure_ascii=False), self.secrets))
            row.clear()
            row.update(safe_row)
            self._write(name + ".receipt.json", json_bytes(row))
            self._save_summary()
        return row

    def check_laws(self):
        law_name = self.config["law_name"]
        law_id = str(self.config["law_id"])
        if not law_id.isdigit():
            raise ValueError("law_id는 법령ID 숫자 문자열입니다.")
        # Can confirm the listed body API even if list forwarding is not yet registered.
        self.request("law_published_body", "lawService.do", "law", {"ID": law_id, "type": "XML"},
                     "law_body", law_name, law_id)
        self.request("law_search", "lawSearch.do", "law", {"query": law_name, "type": "XML", "display": "50", "page": "1"}, "law_list")
        adm = self.request("admrul_search", "lawSearch.do", "admrul",
                           {"query": self.config["admrul_name"], "type": "XML", "display": "50", "page": "1"}, "admrul_list")
        matches = [x for x in adm.get("rows", []) if x["current"] == "현행"
                   and normalized(x["name"]) == normalized(self.config["admrul_name"])]
        if len(matches) == 1 and matches[0]["serial"].isdigit():
            self.request("admrul_body", "lawService.do", "admrul", {"ID": matches[0]["serial"], "type": "XML"},
                         "admrul_body", self.config["admrul_name"], matches[0]["serial"])
        else:
            self.records["admrul_body"] = {"name": "admrul_body", "status": "SKIPPED_NO_UNIQUE_CURRENT_ID",
                                           "reason": "목록의 정확한 현행 항목 한 건이 필요. 임의 ID를 선택하지 않음"}
        # Current unpinned LawGoAdapter uses this endpoint; explicitly not a fallback.
        self.request("law_effective_body", "lawService.do", "eflaw", {"ID": law_id, "type": "XML"},
                     "law_body", law_name, law_id)
        self._save_summary()
        return list(self.records.values())

    def table(self):
        columns = ("name", "status", "http_status", "received_bytes", "effective_date", "reason")
        data = [{k: row.get(k, "") for k in columns} for row in self.records.values()]
        buf = io.StringIO()
        writer = csv.DictWriter(buf, fieldnames=columns)
        writer.writeheader()
        writer.writerows(data)
        path = self.out / "results.csv"
        path.write_text(buf.getvalue(), encoding="utf-8-sig")
        path.chmod(0o600)
        for row in data:
            print(" | ".join(str(row[k]) for k in columns))
        print("결과 위치:", self.out.resolve())
        return data

    def export_to_dataiku(self, folder_id):
        import dataiku
        if not folder_id:
            raise ValueError("Managed Folder ID를 입력하세요.")
        folder = dataiku.Folder(folder_id)
        uploaded = []
        # Use API streams for container/S3 compatibility; no get_path assumption.
        for path in sorted(self.out.iterdir()):
            if path.is_file() and path.suffix in {".json", ".csv", ".gz"}:
                remote = self.run_id + "/" + path.name
                folder.upload_data(remote, path.read_bytes())
                with folder.get_download_stream(remote) as stream:
                    if digest(stream.read()) != digest(path.read_bytes()):
                        raise RuntimeError("Managed Folder 업로드 해시 불일치: " + path.name)
                uploaded.append(remote)
        print("Managed Folder 저장 및 재읽기 검증:", len(uploaded), "파일 /", self.run_id)
        return uploaded


## 3. 인증값 입력과 진단 준비

In [ ]:
# [셀 3] 인증값은 입력창에만 입력합니다. 셀 코드에 붙여 넣지 마세요.
from getpass import getpass
if "check" not in globals():
    _law_oc = os.environ.get("LAW_OC") or getpass("법제처에 등록된 LAW_OC: ")
    _cookie = (os.environ.get("LAW_RELAY_COOKIE") or getpass("현재 중계 인증 Cookie: ")) if USE_RELAY_COOKIE else ""
    _making_oc = (os.environ.get("LAWMAKING_OC") or getpass("입법센터에 등록된 OC: ")) if RUN_NOTICES else ""
    check = RelayCheck(CONFIG, _law_oc, _making_oc, _cookie)
    del _law_oc, _making_oc, _cookie
    print("진단 준비:", check.run_id, " / 모드:", check.mode)
else:
    print("기존 진단을 유지합니다:", check.run_id)
    print("설정/인증을 바꾸어 새로 확인하려면 결과 저장 후 Kernel > Restart를 실행하세요.")


## 4. 선택: 중계 운영 주소 조회

In [ ]:
# [셀 4] 선택: 운영계 IP 조회. 기본은 실행하지 않습니다.
if RUN_IP_LOOKUP:
    check.request("ip_lookup", "ip", "", {"EXTR_INSN_SECT_CODE": EXTR_INSN_SECT_CODE}, "ip")
    print("IP 조회 응답은 결과 폴더에 저장했습니다. 성공코드/활성 주소/환경코드를 담당자 명세로 확인하세요.")
    print("운영 주소 확정 전에는 다음 법령 호출 셀을 실행하지 마세요.")
else:
    print("IP 조회 건너뜀: 개발계 주소 또는 이미 확인한 주소 사용")


## 5. 법령과 행정규칙 확인

In [ ]:
# [셀 5] 법령/행정규칙 최대 5회. 쿠키/키/전체 응답은 출력하지 않습니다.
# 운영 IP 조회 중이면 이 셀은 실행하지 않습니다. 주소 확정 후 설정을 바꾸고 새 커널로 시작합니다.
if RUN_IP_LOOKUP:
    print("주소 조회 모드: 법령 호출 건너뜀. 주소 확정 후 RUN_IP_LOOKUP=False로 새 진단을 시작하세요.")
else:
    check.check_laws()
    print("법령/행정규칙 진단 종료. 셀 7의 각 상태를 확인하세요.")


## 6. 선택: 입법예고·행정예고 확인

In [ ]:
# [셀 6] 선택: 사진1의 행정예고와 입법예고 목록 수신 (각 1회)
if RUN_NOTICES and not RUN_IP_LOOKUP:
    check.request("administrative_notice", "ptcpAdmPp.xml", "",
                  {"lsClsCd": "BA0103", "closing": "Y", "asndOfiNm": "금융위원회",
                   "admRulNm": "지주", "pageIndex": "1", "pageSize": "10"}, "notice")
    check.request("legislative_notice", "ogLmPp.xml", "",
                  {"pageIndex": "1", "pageSize": "10"}, "notice")
    print("예고 XML은 현장 명세 대조 전까지 REVIEW_REQUIRED입니다. 예고안은 현행 법령이 아닙니다.")
else:
    print("예고 API 건너뜀")


## 7. 결과표 확인

In [ ]:
# [셀 7] 성공/실패를 구분한 결과표와 CSV 생성
_result_rows = check.table()
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(_result_rows))
except ImportError:
    pass  # 위의 텍스트 표와 CSV는 이미 생성됨


## 8. 결과를 Dataiku 폴더에 저장

In [ ]:
# [셀 8] Managed Folder에 저장 후 다시 읽어 SHA-256 확인
if MANAGED_FOLDER_ID:
    _saved_files = check.export_to_dataiku(MANAGED_FOLDER_ID)
else:
    print("아직 Managed Folder에 저장하지 않았습니다.")
    print("셀 1의 MANAGED_FOLDER_ID를 입력하고 셀 1과 셀 8을 다시 실행하세요. API는 재호출되지 않습니다.")
